# Hands-On Machine Learning with PyTorch (Aurelion Geron)

## 1) Building Neural Networks with PyTorch

### 1. PyTorch Fundamentals


In [2]:
import torch 

X = torch.tensor([[1.,2.,3.], [4.,5.,6.]])  # shape + dtype, can live on GPU + supports auto-differentiaton
X

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [3]:
# some operations on tensors
print(f"Tensor X is {X}.")
print(f"The maximum value of X over last dimension is {X.max(dim=-1)}")
print(f"The mean value of X over the last dimension is {X.mean(dim=-1)}")
print(f"The absolute value of X is {X.abs()}")
print(f"The cos(X) is {X.cos()}")
print(f"The exp(X) is {X.exp()}")
print(f"The square root of X is {X.sqrt()}")

Tensor X is tensor([[1., 2., 3.],
        [4., 5., 6.]]).
The maximum value of X over last dimension is torch.return_types.max(
values=tensor([3., 6.]),
indices=tensor([2, 2]))
The mean value of X over the last dimension is tensor([2., 5.])
The absolute value of X is tensor([[1., 2., 3.],
        [4., 5., 6.]])
The cos(X) is tensor([[ 0.5403, -0.4161, -0.9900],
        [-0.6536,  0.2837,  0.9602]])
The exp(X) is tensor([[  2.7183,   7.3891,  20.0855],
        [ 54.5981, 148.4132, 403.4288]])
The square root of X is tensor([[1.0000, 1.4142, 1.7321],
        [2.0000, 2.2361, 2.4495]])


#### All these tensors save floats in 32bits by default (in contrast to NumPy).

In [4]:
import numpy as np

# We can convert a NumPy array into a PyTorch tensor efficiently

a = np.array([[1.,2.,3.], [4.,5.,6.]])
t = torch.tensor(a, dtype=torch.float32)

print(f"NumPy array {a} is 64bit.")
print(f"PyTorch tensor {t} is 32bit.")

# Or alternatively:

t_prime = torch.FloatTensor(a)
print(f"PyTorch tensor {t_prime} is 32bit as well.")

NumPy array [[1. 2. 3.]
 [4. 5. 6.]] is 64bit.
PyTorch tensor tensor([[1., 2., 3.],
        [4., 5., 6.]]) is 32bit.
PyTorch tensor tensor([[1., 2., 3.],
        [4., 5., 6.]]) is 32bit as well.


In [5]:
# We can call in-place operations which saves us some memory and speeds up models

t_prime.sqrt_()
print(f"Now t_prime is {t_prime}.")

Now t_prime is tensor([[1.0000, 1.4142, 1.7321],
        [2.0000, 2.2361, 2.4495]]).


In [7]:
# Hardware acceleration

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device is now {device}.")

# We can copy tensors to GPU 
t = t.to(device)

print(f"t is now on GPU: {t}")
print(f"t.device gives {t.device}")

# We can directly create the tensor on the GPU 

M = torch.tensor([[1.,2.,3.], [4.,5.,6.]], device=device)
print(f"M is {M}.")

Device is now cuda.
t is now on GPU: tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')
t.device gives cuda:0
M is tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0').


In [ ]:
# Autograd => reverse-mode auto-differentiation

x = torch.tensor(5.0, requires_grad=True)
f = x**2        # f is of type tensor(25., grad_fn=<PowBackward0>)
f.backward()    # apply backpropagation 

print(f"Gradient of f wrt. x is {x.grad}")
print(f"f has function type {f.grad_fn}")

# PyTorch create new computation graph on the fly during each forward pass 

# What if 2 tensors use same variable? Their gradients are summed up:

a = torch.tensor(5., requires_grad=True)
b = a**2
c = a**3

b.backward()
c.backward()

print(f"Gradients are summed up: {a.grad}")

# Perform gradient step by temporarily disabling GD tracking 

lr = 0.1

with torch.no_grad():
    a -= lr * a.grad

# Alternatively using detach(): 
a_detached = a.detach()
a_detached -= lr * a.grad


# Before repeating FP-BP-GD-step, it is essential to zero out the gradients of every model parameter 

a.grad.zero_()          # done in the loop outside of no_grad

Gradient of f wrt. x is 10.0
f has function type <PowBackward0 object at 0x7c64d809e590>
Gradients are summed up: 85.0
